In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))
from utils.load_utils import load_parameter_results

In [2]:
_PROJ     = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita"
RESULTS_DIR = f"{_PROJ}/optimizations/dots"

FOLDS   = [0, 1, 2, 3]
LAMBDAS = [0.01, 0.1, 1.0, 10.0, 100.0, 150.0, 200.0]

In [3]:
# Lambda sweep
df_lambda = load_parameter_results(RESULTS_DIR, "lambda", LAMBDAS, folds=FOLDS)

Total windows loaded: 1218


In [4]:
df_lambda["no_edits"] = df_lambda["n_edits"] == 0

no_edits_summary = (
    df_lambda.groupby("lambda")["no_edits"]
    .agg(n_total="count", n_no_edits="sum")
    .assign(pct_no_edits=lambda x: 100 * x["n_no_edits"] / x["n_total"])
)
print(no_edits_summary.to_string())

        n_total  n_no_edits  pct_no_edits
lambda                                   
0.01        174           0      0.000000
0.10        174           0      0.000000
1.00        174           0      0.000000
10.00       174           0      0.000000
100.00      174           5      2.873563
150.00      174          26     14.942529
200.00      174          44     25.287356


In [5]:
df_lambda["dot_diff"] = df_lambda["dot15_edited"] - df_lambda["dot15_orig"]
df_lambda["no_improvement"] = df_lambda["dot_diff"] <= 0

no_improvement_summary = (
    df_lambda.groupby("lambda")["no_improvement"]
    .agg(n_total="count", n_no_improvement="sum")
    .assign(pct_no_improvement=lambda x: 100 * x["n_no_improvement"] / x["n_total"])
)
print(no_improvement_summary.to_string())

        n_total  n_no_improvement  pct_no_improvement
lambda                                               
0.01        174                 0            0.000000
0.10        174                 0            0.000000
1.00        174                 0            0.000000
10.00       174                 0            0.000000
100.00      174                 5            2.873563
150.00      174                26           14.942529
200.00      174                44           25.287356


In [6]:
success_rate = (
    df_lambda.assign(success=~df_lambda["no_edits"] & ~df_lambda["no_improvement"])
    .groupby("lambda")["success"]
    .agg(n_total="count", n_success="sum")
    .assign(pct_success=lambda x: 100 * x["n_success"] / x["n_total"])
)
print(success_rate.to_string())

        n_total  n_success  pct_success
lambda                                 
0.01        174        174   100.000000
0.10        174        174   100.000000
1.00        174        174   100.000000
10.00       174        174   100.000000
100.00      174        169    97.126437
150.00      174        148    85.057471
200.00      174        130    74.712644


In [7]:
df_lambda["CTCFs_num"] = df_lambda["CTCFs_num_hi"] + df_lambda["CTCFs_num_lo"]

summary = (
    df_lambda.groupby("lambda")
    .agg(
        n_success        = ("n_edits",          "count"),
        mean_n_edits     = ("n_edits",          "mean"),
        mean_dot_score_diff  = ("dot_diff",        "mean"),
        mean_CTCFs       = ("CTCFs_num",         "mean"),
    )
    .round(3)
)
print(summary.to_string())

        n_success  mean_n_edits  mean_dot_score_diff  mean_CTCFs
lambda                                                          
0.01          174      1136.333                0.557       1.856
0.10          174      1135.943                0.553       1.661
1.00          174      1036.741                0.551       1.960
10.00         174       596.621                0.551       1.914
100.00        174       169.425                0.530       2.259
150.00        174       108.178                0.461       2.063
200.00        174        48.902                0.410       2.011
